<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/rf__baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 50.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, matthews_corrcoef
import warnings

# Silence Python warnings and RDKit C++ backend errors
warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')

# Initialize the Morgan Generator once globally for speed
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Function to safely extract unique 2D molecules and topological features
def prepare_2d_data(sdf_path, label):
    supplier = Chem.SDMolSupplier(sdf_path, removeHs=False)
    data = []
    seen_ids = set()

    for idx, mol in enumerate(supplier):
        if mol is None:
            continue

        # Robust ID assignment to avoid dropping unique molecules
        if mol.HasProp("Parent_ID"):
            mol_id = mol.GetProp("Parent_ID")
        elif mol.HasProp("_Name") and mol.GetProp("_Name").strip() != "":
            mol_id = mol.GetProp("_Name")
        else:
            mol_id = f"Unknown_Mol_{idx}"

        if mol_id in seen_ids:
            continue
        seen_ids.add(mol_id)

        # Remove hydrogens for 2D topological processing
        mol_2d = Chem.RemoveHs(mol)

        # Compute Scaffolds and Morgan Fingerprints
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol_2d, includeChirality=False)

        # GroupKFold will group all completely acyclic molecules into one fold.

        # Compute Morgan Fingerprints using the generator
        fp = mfpgen.GetFingerprint(mol_2d)
        fp_array = np.zeros((0,), dtype=np.int8)
        Chem.DataStructs.ConvertToNumpyArray(fp, fp_array)

        data.append({
            'Parent_ID': mol_id,
            'Scaffold': scaffold,
            'Fingerprint': fp_array,
            'Label': label
        })
    return data

print("Loading datasets and computing 2D features...")

actives_data = prepare_2d_data("Training_Actives_3D.sdf", 1)
negatives_data = prepare_2d_data("Training_Negatives_3D.sdf", 0)

# Combine into a single DataFrame
df = pd.DataFrame(actives_data + negatives_data)
print(f"Total Unique Molecules for Baseline: {len(df)} (Actives: {sum(df['Label']==1)}, Inactives: {sum(df['Label']==0)})\n")

X = np.stack(df['Fingerprint'].values)
y = df['Label'].values
groups = df['Scaffold'].values

# Random Forest Setup
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
gkf = GroupKFold(n_splits=5)

roc_aucs, pr_aucs, f1_scores, mccs = [], [], [], []

print("Executing 5-Fold Bemis-Murcko Scaffold Split CV...")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    rf.fit(X_train, y_train)

    y_probs = rf.predict_proba(X_val)[:, 1]
    y_preds = rf.predict(X_val)

    #  Check if the fold actually has both classes to prevent math domain errors
    if len(np.unique(y_val)) > 1:
        # 1. ROC-AUC
        fold_roc = roc_auc_score(y_val, y_probs)

        # 2. PR-AUC
        precision, recall, _ = precision_recall_curve(y_val, y_probs)
        fold_pr = auc(recall, precision)

        # 3. F1 Score
        fold_f1 = f1_score(y_val, y_preds)

        # 4. MCC
        fold_mcc = matthews_corrcoef(y_val, y_preds)
    else:
        # If the fold only has 1 class, metrics are undefined
        fold_roc, fold_pr, fold_f1, fold_mcc = np.nan, np.nan, np.nan, np.nan
        print(f"  -> Warning: Fold {fold+1} only contains one class. Metrics set to NaN.")

    roc_aucs.append(fold_roc)
    pr_aucs.append(fold_pr)
    f1_scores.append(fold_f1)
    mccs.append(fold_mcc)

    print(f"Fold {fold+1} | ROC-AUC: {fold_roc:.3f} | PR-AUC: {fold_pr:.3f} | F1: {fold_f1:.3f} | MCC: {fold_mcc:.3f}")

print("\nFinal 2D Baseline Performance (Morgan FP + Random Forest)")
print(f"Average ROC-AUC:  {np.nanmean(roc_aucs):.3f} ± {np.nanstd(roc_aucs):.3f}")
print(f"Average PR-AUC:   {np.nanmean(pr_aucs):.3f} ± {np.nanstd(pr_aucs):.3f}")
print(f"Average F1-Score: {np.nanmean(f1_scores):.3f} ± {np.nanstd(f1_scores):.3f}")
print(f"Average MCC:      {np.nanmean(mccs):.3f} ± {np.nanstd(mccs):.3f}")

Loading datasets and computing 2D features...
Total Unique Molecules for Baseline: 137 (Actives: 83, Inactives: 54)

Executing 5-Fold Bemis-Murcko Scaffold Split CV...
Fold 1 | ROC-AUC: 0.957 | PR-AUC: 0.991 | F1: 0.905 | MCC: 0.677
Fold 2 | ROC-AUC: 0.977 | PR-AUC: 0.990 | F1: 0.919 | MCC: 0.764
Fold 3 | ROC-AUC: 0.934 | PR-AUC: 0.970 | F1: 0.865 | MCC: 0.574
Fold 4 | ROC-AUC: 0.994 | PR-AUC: 0.993 | F1: 0.889 | MCC: 0.800
Fold 5 | ROC-AUC: 0.929 | PR-AUC: 0.908 | F1: 0.750 | MCC: 0.586

Final 2D Baseline Performance (Morgan FP + Random Forest)
Average ROC-AUC:  0.958 ± 0.025
Average PR-AUC:   0.970 ± 0.032
Average F1-Score: 0.865 ± 0.060
Average MCC:      0.680 ± 0.091


In [5]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator
from rdkit import RDLogger

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score, matthews_corrcoef
import warnings

# Silence Python warnings and RDKit C++ backend errors
warnings.filterwarnings('ignore')
RDLogger.DisableLog('rdApp.*')

# Initialize the Morgan Generator once globally for speed
mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Function to safely extract unique 2D molecules and topological features
def prepare_2d_data(sdf_path, label):
    supplier = Chem.SDMolSupplier(sdf_path, removeHs=False)
    data = []
    seen_ids = set()

    for idx, mol in enumerate(supplier):
        if mol is None:
            # Corrupted conformers are skipped cleanly here
            continue

        # Robust ID assignment to avoid dropping unique molecules
        if mol.HasProp("Parent_ID"):
            mol_id = mol.GetProp("Parent_ID")
        elif mol.HasProp("_Name") and mol.GetProp("_Name").strip() != "":
            mol_id = mol.GetProp("_Name")
        else:
            mol_id = f"Unknown_Mol_{idx}"

        if mol_id in seen_ids:
            continue
        seen_ids.add(mol_id)

        # Remove hydrogens for safe 2D topological processing
        mol_2d = Chem.RemoveHs(mol)

        # Compute Scaffolds
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol_2d, includeChirality=False)

        # Compute Morgan Fingerprints
        fp = mfpgen.GetFingerprint(mol_2d)
        fp_array = np.zeros((0,), dtype=np.int8)
        Chem.DataStructs.ConvertToNumpyArray(fp, fp_array)

        data.append({
            'Parent_ID': mol_id,
            'Scaffold': scaffold,
            'Fingerprint': fp_array,
            'Label': label
        })
    return data

print("Loading datasets and computing 2D features...")

actives_data = prepare_2d_data("Training_Actives_3D.sdf", 1)
negatives_data = prepare_2d_data("Training_Negatives_3D.sdf", 0)

# Combine into a single DataFrame
df = pd.DataFrame(actives_data + negatives_data)
print(f"Total Unique Molecules for Baseline: {len(df)} (Actives: {sum(df['Label']==1)}, Inactives: {sum(df['Label']==0)})\n")

X = np.stack(df['Fingerprint'].values)
y = df['Label'].values
groups = df['Scaffold'].values

# Random Forest Setup
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
gkf = GroupKFold(n_splits=5)

# Y Randomization
N_ITERATIONS = 1000
np.random.seed(42)

# Storage for global summary comparison
all_rand_rocs, all_rand_prs, all_rand_f1s, all_rand_mccs = [], [], [], []

print(f"Executing Y-Randomization Validation ({N_ITERATIONS} Permutations)...")

for iteration in range(N_ITERATIONS):
    # Generating a random index permutation array
    permuted_indices = np.random.permutation(len(y))
    y_scrambled = y[permuted_indices]

    roc_aucs, pr_aucs, f1_scores, mccs = [], [], [], []

    # Run the exact same GroupKFold workflow on scrambled labels
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_scrambled, groups=groups)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y_scrambled[train_idx], y_scrambled[val_idx]

        # Train model on scrambled labels
        rf.fit(X_train, y_train)

        y_probs = rf.predict_proba(X_val)[:, 1]
        y_preds = rf.predict(X_val)

        # Check if the fold actually has both classes to prevent math domain errors
        if len(np.unique(y_val)) > 1:
            fold_roc = roc_auc_score(y_val, y_probs)
            precision, recall, _ = precision_recall_curve(y_val, y_probs)
            fold_pr = auc(recall, precision)
            fold_f1 = f1_score(y_val, y_preds)
            fold_mcc = matthews_corrcoef(y_val, y_preds)
        else:
            fold_roc, fold_pr, fold_f1, fold_mcc = np.nan, np.nan, np.nan, np.nan

        roc_aucs.append(fold_roc)
        pr_aucs.append(fold_pr)
        f1_scores.append(fold_f1)
        mccs.append(fold_mcc)

    # Calculate means for this specific scrambled iteration
    iter_roc = np.nanmean(roc_aucs)
    iter_pr  = np.nanmean(pr_aucs)
    iter_f1  = np.nanmean(f1_scores)
    iter_mcc = np.nanmean(mccs)

    # Only print every 100 iterations to prevent terminal lag
    if (iteration + 1) % 100 == 0 or iteration == 0:
        print(f"Run {iteration+1}/{N_ITERATIONS} Mean Metrics | ROC-AUC: {iter_roc:.3f} | PR-AUC: {iter_pr:.3f} | F1: {iter_f1:.3f} | MCC: {iter_mcc:.3f}")

    # Save the iteration results for global calculation
    all_rand_rocs.append(iter_roc)
    all_rand_prs.append(iter_pr)
    all_rand_f1s.append(iter_f1)
    all_rand_mccs.append(iter_mcc)

# Comparison
print("Y-Randomization Performance Report:")
print(f"Average ROC-AUC:  {np.nanmean(all_rand_rocs):.3f} ± {np.nanstd(all_rand_rocs):.3f}")
print(f"Average PR-AUC:   {np.nanmean(all_rand_prs):.3f} ± {np.nanstd(all_rand_prs):.3f}")
print(f"Average F1-Score: {np.nanmean(all_rand_f1s):.3f} ± {np.nanstd(all_rand_f1s):.3f}")
print(f"Average MCC:      {np.nanmean(all_rand_mccs):.3f} ± {np.nanstd(all_rand_mccs):.3f}")


Loading datasets and computing 2D features...
Total Unique Molecules for Baseline: 137 (Actives: 83, Inactives: 54)

Executing Y-Randomization Validation (1000 Permutations)...
Run 1/1000 Mean Metrics | ROC-AUC: 0.469 | PR-AUC: 0.571 | F1: 0.643 | MCC: 0.023
Run 100/1000 Mean Metrics | ROC-AUC: 0.383 | PR-AUC: 0.559 | F1: 0.593 | MCC: -0.240
Run 200/1000 Mean Metrics | ROC-AUC: 0.494 | PR-AUC: 0.628 | F1: 0.590 | MCC: -0.049
Run 300/1000 Mean Metrics | ROC-AUC: 0.416 | PR-AUC: 0.553 | F1: 0.665 | MCC: -0.031
Run 400/1000 Mean Metrics | ROC-AUC: 0.508 | PR-AUC: 0.631 | F1: 0.685 | MCC: 0.001
Run 500/1000 Mean Metrics | ROC-AUC: 0.445 | PR-AUC: 0.557 | F1: 0.666 | MCC: -0.029
Run 600/1000 Mean Metrics | ROC-AUC: 0.369 | PR-AUC: 0.566 | F1: 0.671 | MCC: -0.014
Run 700/1000 Mean Metrics | ROC-AUC: 0.367 | PR-AUC: 0.518 | F1: 0.609 | MCC: -0.108
Run 800/1000 Mean Metrics | ROC-AUC: 0.534 | PR-AUC: 0.639 | F1: 0.668 | MCC: 0.004
Run 900/1000 Mean Metrics | ROC-AUC: 0.598 | PR-AUC: 0.684 | F1